# PhoBERT Improved Training\n
\n
Notebook Kaggle này clone/pull branch `feature/improve-training-v2`, chạy hai encoder với lr=2e-5, chọn checkpoint theo Combined F1, evaluate ensemble top-3 và đóng gói kết quả.

In [ ]:
import os, sys, subprocess, shutil, json\n
\n
REPO_URL = 'https://github.com/vudinhminh08/NLP-project-master-study.git'\n
BRANCH = 'feature/improve-training-v2'\n
PROJECT_DIR = '/kaggle/working/absa-project'\n
\n
if os.path.exists(os.path.join(PROJECT_DIR, '.git')):\n
    subprocess.check_call(['git', '-C', PROJECT_DIR, 'fetch', 'origin', BRANCH])\n
    subprocess.check_call(['git', '-C', PROJECT_DIR, 'checkout', BRANCH])\n
    subprocess.check_call(['git', '-C', PROJECT_DIR, 'pull', '--ff-only', 'origin', BRANCH])\n
else:\n
    if os.path.exists(PROJECT_DIR):\n
        shutil.rmtree(PROJECT_DIR)\n
    subprocess.check_call(['git', 'clone', '--branch', BRANCH, REPO_URL, PROJECT_DIR])\n
\n
os.chdir(PROJECT_DIR)\n
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'])\n
\n
sys.path.insert(0, os.path.join(PROJECT_DIR, 'code', 'phobert'))\n
sys.path.insert(0, os.path.join(PROJECT_DIR, 'code', 'data_processing'))\n
print('Working dir:', os.getcwd())\n
print('Branch:', subprocess.check_output(['git', 'branch', '--show-current'], text=True).strip())

In [ ]:
import torch\n
print('CUDA available:', torch.cuda.is_available())\n
if torch.cuda.is_available():\n
    print('GPU:', torch.cuda.get_device_name(0))\n
    torch.cuda.empty_cache()

In [ ]:
from run_experiment import main\n
\n
cls_metrics = main(\n
    encoder_option='cls_only',\n
    use_amp=True,\n
    lr=2e-5,\n
    max_epochs=40,\n
    early_stop_patience=10,\n
)\n
print('cls_only primary Combined F1:', cls_metrics['primary_combined_f1'])

In [ ]:
import torch\n
torch.cuda.empty_cache()\n
\n
concat_metrics = main(\n
    encoder_option='concat_4_layers',\n
    use_amp=True,\n
    lr=2e-5,\n
    max_epochs=40,\n
    early_stop_patience=10,\n
)\n
print('concat_4_layers primary Combined F1:', concat_metrics['primary_combined_f1'])

In [ ]:
import json, os\n
import matplotlib.pyplot as plt\n
\n
RUNS = {\n
    'cls_only lr=2e-5': 'outputs/results_cls_only_lr2e5/training_history.json',\n
    'concat_4_layers lr=2e-5': 'outputs/results_lr2e5/training_history.json',\n
}\n
\n
fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n
for label, path in RUNS.items():\n
    if not os.path.exists(path):\n
        print('Missing history:', path)\n
        continue\n
    h = json.load(open(path, encoding='utf-8'))\n
    epochs = range(1, len(h['train_loss']) + 1)\n
    axes[0].plot(epochs, h['train_loss'], marker='o', label=f'{label} train')\n
    axes[0].plot(epochs, h['dev_loss'], marker='s', label=f'{label} dev')\n
    axes[1].plot(epochs, h['dev_combined_f1'], marker='o', label=label)\n
    axes[1].axvline(h['best_epoch'], linestyle='--', alpha=0.35)\n
\n
axes[0].set_title('Loss')\n
axes[0].set_xlabel('Epoch')\n
axes[0].grid(alpha=0.3)\n
axes[0].legend()\n
axes[1].set_title('Dev Combined F1')\n
axes[1].set_xlabel('Epoch')\n
axes[1].axhline(0.7732, color='red', linestyle=':', label='SOTA 0.7732')\n
axes[1].grid(alpha=0.3)\n
axes[1].legend()\n
plt.tight_layout()\n
os.makedirs('outputs/eda', exist_ok=True)\n
plt.savefig('outputs/eda/learning_curve_improved.png', dpi=150, bbox_inches='tight')\n
plt.show()

In [ ]:
import os, json\n
import pandas as pd\n
\n
def add_metrics(rows, name, path):\n
    if not os.path.exists(path):\n
        return\n
    m = json.load(open(path, encoding='utf-8'))\n
    rows.append({\n
        'Run': name,\n
        'ACD F1': m['macro_acd_f1'],\n
        'SPC F1': m['macro_spc_f1'],\n
        'Combined F1': m['macro_combined_f1'],\n
    })\n
\n
rows = []\n
add_metrics(rows, 'Baseline old cls_only lr=1e-4', 'outputs/results/phobert_best_single/results_cls_only/phobert_test_metrics.json')\n
add_metrics(rows, 'Baseline old concat_4_layers', 'outputs/results/phobert_no_vncorenlp/results/phobert_test_metrics.json')\n
add_metrics(rows, 'New cls_only lr=2e-5 single', 'outputs/results_cls_only_lr2e5/phobert_test_metrics.json')\n
add_metrics(rows, 'New cls_only lr=2e-5 ensemble', 'outputs/results_cls_only_lr2e5/phobert_test_ensemble_metrics.json')\n
add_metrics(rows, 'New concat_4_layers lr=2e-5 single', 'outputs/results_lr2e5/phobert_test_metrics.json')\n
add_metrics(rows, 'New concat_4_layers lr=2e-5 ensemble', 'outputs/results_lr2e5/phobert_test_ensemble_metrics.json')\n
rows.append({'Run': 'SOTA ds4v 2022', 'ACD F1': 0.8255, 'SPC F1': None, 'Combined F1': 0.7732})\n
\n
df = pd.DataFrame(rows)\n
display(df)\n
candidates = df[df['Run'].str.startswith('New ')].dropna(subset=['Combined F1'])\n
primary = candidates.sort_values('Combined F1', ascending=False).iloc[0]\n
print('PRIMARY_RESULT =', primary['Run'])\n
print('PRIMARY_COMBINED_F1 =', f"{primary['Combined F1']:.4f}")\n
df.to_csv('outputs/results_improved_comparison.csv', index=False)

In [ ]:
import shutil\n
zip_path = shutil.make_archive('/kaggle/working/phobert_improved_results', 'zip', 'outputs')\n
print('Zip:', zip_path)